In [ ]:
!pip install langchain_ollama

!apt-get update
!apt-get install -y graphviz graphviz-dev
!pip install pygraphviz

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,969 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.0 MB]
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 Packages [38.8 kB]
Hit:10 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,929 kB]
0% [Waiting for headers] [Waiting for headers]

In [ ]:
from google.colab import userdata
import os
langsmith_api_key = userdata.get('LANGSMITH')

os.environ["LANGSMITH_TRACING"]="true"
os.environ["LANGSMITH_ENDPOINT"]="https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"]=langsmith_api_key
os.environ["LANGSMITH_PROJECT"]="Codeing_Agent_2"

In [ ]:
import os
import json
import subprocess
from typing import List, Literal

from typing_extensions import TypedDict

from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
from langchain_ollama import ChatOllama

from langgraph.graph import StateGraph, START, END
from langsmith import traceable

In [ ]:
!apt-get install -y zstd

In [ ]:
!apt-get update
!apt-get install -y pciutils lshw

In [ ]:
!lspci | grep -i nvidia

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time

subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

In [ ]:
!ollama pull qwen2.5-coder:7b

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen2.5-coder:7b",
    temperature=0.2,
)

In [ ]:
class AgentState(TypedDict):
    messages: List[BaseMessage]
    user_request: str
    plan: str
    context: str
    code_action: str
    last_tool_result: str
    status: str
    retry_count: int
    final_answer: str


MAX_RETRIES = 5

In [ ]:
def ask_user_approval(message: str) -> bool:

    answer = input(f"\n[APPROVAL REQUIRED] {message} (y/n): ").strip().lower()
    return answer == "y"

In [ ]:
def _edit_file(filename: str, find_str: str, replace_str: str) -> str:

    if not os.path.exists(filename) and find_str == "":
        with open(filename, "w") as f:
            f.write(replace_str)
        return f"Created new file: {filename}"

    try:
        with open(filename, "r") as f:
            content = f.read()

        if find_str in content:
            new_content = content.replace(find_str, replace_str)
            with open(filename, "w") as f:
                f.write(new_content)
            return f"Successfully edited: {filename}"
        else:
            return f"Error: find_str not found in {filename}"
    except FileNotFoundError:
        return f"Error: File {filename} not found and find_str is not empty."

In [ ]:
def _run_command(command: str, working_dir: str) -> tuple[str, int]:

    try:
        process = subprocess.Popen(
            command,
            shell=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            cwd=working_dir,
        )
        output, _ = process.communicate()
        error_code = process.returncode

        if len(output) > 2000:
            output = output[:1000] + "\n\n[...content clipped...]\n\n" + output[-1000:]

        return output, error_code
    except Exception as e:
        return str(e), 1

In [ ]:
def _list_directory(path: str = ".") -> str:

    try:
        items = os.listdir(path)
        if not items:
            return f"Directory '{path}' is empty."

        result = f"Contents of directory '{path}':\n"
        for item in items:
            full_path = os.path.join(path, item)
            item_type = "Directory" if os.path.isdir(full_path) else "File"
            result += f"- {item} ({item_type})\n"
        return result.strip()
    except FileNotFoundError:
        return f"Error: Directory '{path}' not found."
    except PermissionError:
        return f"Error: Permission denied to access '{path}'."
    except Exception as e:
        return f"Error listing directory '{path}': {str(e)}"

In [ ]:

def _read_file_content(path: str) -> str:

    try:
        with open(path, "r", encoding="utf-8") as f:
            content = f.read()
        if len(content) > 2000:
            content = content[:1000] + "\n\n[...content clipped...]\n\n" + content[-1000:]
        return content
    except FileNotFoundError:
        return f"Error: File '{path}' not found."
    except PermissionError:
        return f"Error: Permission denied to access '{path}'."
    except UnicodeDecodeError:
        return f"Error: Unable to decode '{path}'. File may be binary."
    except Exception as e:
        return f"Error reading file '{path}': {str(e)}"

In [ ]:
@tool
def edit_file(filename: str, find_str: str, replace_str: str) -> str:
    """
    Apply a diff to a file by replacing occurrences of find_str with
    replace_str. Pass find_str='' to create a new file with replace_str
    as its contents.
    """
    print(f"\n[TOOL] edit_file → {filename}")
    if find_str:
        print(f"  find:    {find_str[:120]!r}")
    print(f"  replace: {replace_str[:120]!r}")

    if not ask_user_approval("Do you want to edit this file?"):
        return "File edit cancelled by user."
    return _edit_file(filename, find_str, replace_str)

In [ ]:
@tool
def run_command(command: str, working_dir: str) -> str:
    """
    Run a shell command in working_dir and return its combined
    stdout/stderr output followed by the exit code.
    """
    print(f"\n[TOOL] run_command → {command}  (cwd={working_dir})")
    if not ask_user_approval("Do you want to execute this command?"):
        return "Command execution cancelled by user. exit_code=0"
    output, code = _run_command(command, working_dir)
    return f"{output}\n[exit_code={code}]"

In [ ]:
@tool
def list_directory(path: str) -> str:
    """List the contents of a directory (files and sub-directories)."""
    print(f"\n[TOOL] list_directory → {path}")
    return _list_directory(path)


In [ ]:
@tool
def read_file_content(path: str) -> str:
    """Read and return the text content of a file (auto-clips if > 2 KB)."""
    print(f"\n[TOOL] read_file_content → {path}")
    return _read_file_content(path)

In [ ]:
lc_tools = [edit_file, run_command, list_directory, read_file_content]
llm_with_tools = llm.bind_tools(lc_tools)

In [ ]:
def _text(msg: BaseMessage) -> str:
    if isinstance(msg.content, str):
        return msg.content
    # some models return a list of content blocks
    return " ".join(
        block.get("text", "") if isinstance(block, dict) else str(block)
        for block in msg.content
    )

## Nodes

### planner


In [ ]:
@traceable(name="node_planner")
def node_planner(state: AgentState) -> AgentState:

    print("\n" + "=" * 60)
    print("PLANNER")
    print("=" * 60)

    system = SystemMessage(content=(
        "You are a senior software engineer acting as a Planner.\n"
        "Your ONLY job is to read the user's request and output a clear, "
        "numbered step-by-step plan that the team will follow.\n"
        "Be concise. Do NOT write any code yet. Do NOT call any tools.\n"
        "Just produce the plan as a numbered list."
    ))
    human = HumanMessage(content=(
        f"User request:\n{state['user_request']}\n\n"
        "Write the step-by-step plan to accomplish this request."
    ))

    response = llm.invoke([system, human])
    plan_text = _text(response)

    print(f"\nPlan:\n{plan_text}")

    return {
        **state,
        "plan": plan_text,
        "messages": state["messages"] + [
            human,
            AIMessage(content=f"[PLAN]\n{plan_text}"),
        ],
        "status": "running",
    }


### Researcher

In [ ]:
@traceable(name="node_researcher")
def node_researcher(state: AgentState) -> AgentState:

    print("\n" + "=" * 60)
    print("RESEARCHER")
    print("=" * 60)

    system = SystemMessage(content=(
        "You are a Researcher agent. Your goal is to gather all context "
        "needed before writing or modifying code.\n"
        "Use list_directory and read_file_content to inspect the project.\n"
        "After gathering enough information, write a concise CONTEXT SUMMARY "
        "that the Writer will use. Do NOT modify any files."
    ))
    human = HumanMessage(content=(
        f"Plan to execute:\n{state['plan']}\n\n"
        "Explore the relevant files/directories and summarise what you find."
    ))


    safe_tools = [list_directory, read_file_content]
    llm_safe = llm.bind_tools(safe_tools)

    messages: List[BaseMessage] = [system, human]
    context_summary = ""


    for _ in range(6):
        response = llm_safe.invoke(messages)
        messages.append(response)

        tool_calls = getattr(response, "tool_calls", [])
        if not tool_calls:
            context_summary = _text(response)
            break

        for tc in tool_calls:
            tool_name = tc["name"]
            tool_args = tc["args"]

            if tool_name == "list_directory":
                result = _list_directory(**tool_args)
            elif tool_name == "read_file_content":
                result = _read_file_content(**tool_args)
            else:
                result = f"Tool {tool_name} not available in Researcher."

            from langchain_core.messages import ToolMessage
            messages.append(ToolMessage(content=result, tool_call_id=tc["id"]))

    print(f"\nContext summary:\n{context_summary}")

    return {
        **state,
        "context": context_summary,
        "messages": state["messages"] + [
            AIMessage(content=f"[CONTEXT]\n{context_summary}"),
        ],
    }


### Writer

In [ ]:
@traceable(name="node_writer")
def node_writer(state: AgentState) -> AgentState:
on list in state['code_action'].

    print("\n" + "=" * 60)
    print("WRITER")
    print("=" * 60)

    retry_note = ""
    if state.get("retry_count", 0) > 0:
        retry_note = (
            f"\n\nPrevious attempt failed. Tool result was:\n"
            f"{state.get('last_tool_result', 'unknown error')}\n"
            "Please adjust your approach and try again."
        )

    system = SystemMessage(content=(
        "You are a Writer agent responsible for producing the exact tool "
        "calls needed to implement the plan.\n"
        "Output a JSON array of actions. Each action has:\n"
        "  {\"tool\": \"<tool_name>\", \"args\": {<args dict>}}\n\n"
        "Available tools: edit_file, run_command, list_directory, read_file_content\n"
        "For edit_file: args = {filename, find_str, replace_str}\n"
        "For run_command: args = {command, working_dir}\n"
        "For list_directory: args = {path}\n"
        "For read_file_content: args = {path}\n\n"
        "Return ONLY the JSON array – no markdown fences, no explanation."
    ))
    human = HumanMessage(content=(
        f"Plan:\n{state['plan']}\n\n"
        f"Context:\n{state['context']}\n"
        f"{retry_note}\n\n"
        "Output the JSON action list now."
    ))

    response = llm.invoke([system, human])
    raw = _text(response).strip()


    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    raw = raw.strip().rstrip("`").strip()

    print(f"\nProposed actions (raw):\n{raw}")

    return {
        **state,
        "code_action": raw,
        "messages": state["messages"] + [
            AIMessage(content=f"[WRITER ACTIONS]\n{raw}"),
        ],
    }

### Executor

In [ ]:
@traceable(name="node_executor")
def node_executor(state: AgentState) -> AgentState:

    print("\n" + "=" * 60)
    print("EXECUTOR")
    print("=" * 60)

    raw_actions = state.get("code_action", "[]")
    results: List[str] = []

    try:
        actions = json.loads(raw_actions)
    except json.JSONDecodeError as e:
        error_msg = f"Error: Writer produced invalid JSON – {e}\nRaw:\n{raw_actions}"
        print(error_msg)
        return {**state, "last_tool_result": error_msg}

    for idx, action in enumerate(actions):
        tool_name = action.get("tool", "")
        args = action.get("args", {})
        print(f"\n[{idx + 1}/{len(actions)}] Executing: {tool_name}({args})")


        if tool_name == "edit_file":
            filename = args.get("filename", "")
            find_str = args.get("find_str", "")
            replace_str = args.get("replace_str", "")

            if find_str:
                print(f"  find:    {find_str[:120]!r}")
            print(f"  replace: {replace_str[:120]!r}")

            if not ask_user_approval(f"Do you want to edit '{filename}'?"):
                result = "File edit cancelled by user."
            else:
                result = _edit_file(filename, find_str, replace_str)
            print(f"  Result: {result}")
            results.append(f"edit_file({filename}): {result}")


        elif tool_name == "run_command":
            command = args.get("command", "")
            working_dir = args.get("working_dir", ".")

            if not ask_user_approval(f"Do you want to run: {command!r}?"):
                result_str = "Command execution cancelled by user. exit_code=0"
            else:
                output, code = _run_command(command, working_dir)
                result_str = f"{output}\n[exit_code={code}]"
            print(result_str)
            results.append(f"run_command({command}): {result_str}")


        elif tool_name == "list_directory":
            path = args.get("path", ".")
            result = _list_directory(path)
            print(result)
            results.append(f"list_directory({path}): {result}")


        elif tool_name == "read_file_content":
            path = args.get("path", "")
            result = _read_file_content(path)
            print(result)
            results.append(f"read_file_content({path}): {result}")

        else:
            msg = f"Unknown tool: {tool_name}"
            print(f"{msg}")
            results.append(msg)

    combined = "\n\n".join(results) if results else "No actions were executed."
    return {
        **state,
        "last_tool_result": combined,
        "messages": state["messages"] + [
            AIMessage(content=f"[EXECUTOR RESULTS]\n{combined}"),
        ],
    }



### Critic

In [ ]:
@traceable(name="node_critic")
def node_critic(state: AgentState) -> AgentState:

    print("\n" + "=" * 60)
    print("CRITIC")
    print("=" * 60)

    retry_count = state.get("retry_count", 0)
    if retry_count >= MAX_RETRIES:
        print(f"Max retries ({MAX_RETRIES}) reached – forcing success.")
        return {**state, "status": "success"}

    system = SystemMessage(content=(
        "You are a strict Critic agent. Your only job is to evaluate whether "
        "the tool execution results indicate SUCCESS or FAILURE.\n\n"
        "Rules:\n"
        "- Any [exit_code=1] or [exit_code=<non-zero>] is a FAILURE.\n"
        "- Any message containing 'Error:' or 'cancelled by user' is a FAILURE.\n"
        "- If all actions produced clean output with exit_code=0, that is SUCCESS.\n"
        "- If no commands were run but files were correctly created/edited, SUCCESS.\n\n"
        "Respond with EXACTLY one word: SUCCESS or RETRY.\n"
        "Then on a new line, write a one-sentence explanation."
    ))
    human = HumanMessage(content=(
        f"Plan:\n{state['plan']}\n\n"
        f"Tool results:\n{state['last_tool_result']}"
    ))

    response = llm.invoke([system, human])
    verdict_text = _text(response).strip()
    print(f"\nCritic verdict:\n{verdict_text}")

    first_word = verdict_text.split()[0].upper() if verdict_text else "RETRY"
    status = "success" if first_word == "SUCCESS" else "retry"

    return {
        **state,
        "status": status,
        "retry_count": retry_count + (1 if status == "retry" else 0),
        "messages": state["messages"] + [
            AIMessage(content=f"[CRITIC] {verdict_text}"),
        ],
    }

### Finalizer

In [ ]:
@traceable(name="node_finalizer")
def node_finalizer(state: AgentState) -> AgentState:

    print("\n" + "=" * 60)
    print("FINALIZER")
    print("=" * 60)

    system = SystemMessage(content=(
        "You are a Finalizer agent. The coding task has been completed.\n"
        "Write a clear, concise summary for the user that explains:\n"
        "1. What was accomplished\n"
        "2. Which files were created or modified\n"
        "3. Which commands were run and their outcomes\n"
        "4. Any important notes or next steps\n"
        "Be friendly and professional."
    ))
    human = HumanMessage(content=(
        f"Original request:\n{state['user_request']}\n\n"
        f"Plan executed:\n{state['plan']}\n\n"
        f"Tool results:\n{state['last_tool_result']}\n\n"
        "Please write the final summary for the user."
    ))

    response = llm.invoke([system, human])
    final = _text(response)

    print(f"\n{'=' * 60}")
    print("  FINAL ANSWER")
    print("=" * 60)
    print(final)

    return {
        **state,
        "final_answer": final,
        "messages": state["messages"] + [AIMessage(content=final)],}

In [ ]:
def route_after_critic(state: AgentState) -> Literal["node_writer", "node_finalizer"]:

    if state.get("status") == "success":
        return "node_finalizer"
    return "node_writer"


In [ ]:
def build_graph() -> "CompiledGraph":

    graph = StateGraph(AgentState)

    # Register nodes
    graph.add_node("node_planner",    node_planner)
    graph.add_node("node_researcher", node_researcher)
    graph.add_node("node_writer",     node_writer)
    graph.add_node("node_executor",   node_executor)
    graph.add_node("node_critic",     node_critic)
    graph.add_node("node_finalizer",  node_finalizer)

    # Fixed edges
    graph.add_edge(START,              "node_planner")
    graph.add_edge("node_planner",     "node_researcher")
    graph.add_edge("node_researcher",  "node_writer")
    graph.add_edge("node_writer",      "node_executor")
    graph.add_edge("node_executor",    "node_critic")

    # Conditional edge from Critic
    graph.add_conditional_edges(
        "node_critic",
        route_after_critic,
        {
            "node_writer":    "node_writer",     # retry loop
            "node_finalizer": "node_finalizer",  # done
        },
    )

    graph.add_edge("node_finalizer", END)

    return graph.compile()

In [ ]:
from IPython.display import Image, display

graph = build_graph()

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Graph visualization skipped:", e)

In [ ]:
def run_agent(user_request: str) -> str:

    initial_state: AgentState = {
        "messages":        [HumanMessage(content=user_request)],
        "user_request":    user_request,
        "plan":            "",
        "context":         "",
        "code_action":     "",
        "last_tool_result": "",
        "status":          "running",
        "retry_count":     0,
        "final_answer":    "",
    }

    graph = build_graph()

    # Stream node-by-node so progress is visible in the terminal
    final_state = None
    for chunk in graph.stream(initial_state, {"recursion_limit": 50}):
        final_state = chunk

    if final_state:
        # The last chunk is a dict keyed by the last node name
        last_node_output = list(final_state.values())[-1]
        return last_node_output.get("final_answer", "Task completed.")

    return "No output produced."

In [ ]:
if __name__ == "__main__":

    user_input = input("How can I help you?\n> ").strip()

    if user_input:
        run_agent(user_input)
    else:
        print("No input provided. Exiting.")